# Cafe Marketing and Retail Analytics

**Recruiter-facing end-to-end analysis · Transactional business analysis and item segmentation · Python 3.12/3.13**

> The cleaned year contains 69,982 bills and 32,654,640 recorded revenue units; two stable menu-item groups score 0.395 silhouette.

## Executive summary

**Objective:** Translate point-of-sale lines into revenue, order, daypart, basket, and stable menu-item insights supported by available fields.

**Data:** 145,830 cafe line items covering 69,982 bills and one year of activity.

**Verified result:** The cleaned year contains 69,982 bills and 32,654,640 recorded revenue units; two stable menu-item groups score 0.395 silhouette.

**Decision supported:** Prioritize menu review, daypart staffing, and carefully tested cross-sell ideas.

The figures, tables, metrics, and execution counts in this notebook are saved outputs from the bundled data.

## 1. Business understanding

**Primary user:** A cafe operator or marketing analyst.

**Decision:** Prioritize menu review, daypart staffing, and carefully tested cross-sell ideas.

**Why it matters:** A technically accurate result is useful only when its error costs, uncertainty, and decision boundary are visible. This project stays within the evidence available in the source data.

## 2. Analytical objective and success criteria

Technical success requires portable execution, explicit data-quality evidence, a justified baseline, leakage-safe validation, task-appropriate metrics, diagnostics, and saved artifacts. Business success requires a specific recommendation supported by the observed result without invented financial impact.

## 3. Reproducible environment

In [1]:
from pathlib import Path
import hashlib, importlib.util, json, os, platform, tempfile, time
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "portfolio-matplotlib-cache"))
import matplotlib, numpy as np, pandas as pd, scipy, sklearn
SLUG = '10-marketing-retail-analysis'
def locate_project():
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = base if base.name == SLUG else base / "projects" / SLUG
        if (candidate / "src" / "analysis.py").is_file(): return candidate
    raise FileNotFoundError(SLUG)
PROJECT_ROOT = locate_project(); DATA_DIR = PROJECT_ROOT / "data"; REPORTS_DIR = PROJECT_ROOT / "reports"
print(pd.Series({"Python": platform.python_version(), "pandas": pd.__version__, "NumPy": np.__version__, "SciPy": scipy.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__}, name="version").to_string())
print(f"\nProject: {PROJECT_ROOT.name}")

Python          3.12.13
pandas            2.2.3
NumPy             2.3.5
SciPy            1.17.0
scikit-learn      1.8.0
Matplotlib       3.10.8

Project: 10-marketing-retail-analysis


## 4. Data provenance and scope

Bundled in the original repository; business identity, geography, currency, and redistribution terms are not documented.

The next cells expose exact files, byte sizes, checksums, schemas, and sample records.

### 4.1 Source-file inventory

In [2]:
rows=[]
for path in sorted(DATA_DIR.iterdir()):
    if path.is_file() and path.name != "README.md": rows.append({"file": path.name, "size_mb": round(path.stat().st_size/1_000_000,3), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:16]})
inventory=pd.DataFrame(rows); print(inventory.to_string(index=False))

                  file  size_mb           sha256
cafe_transactions.xlsx    8.282 ea3c0b8e8a7d9d4b


### 4.2 Raw-record and schema preview

In [3]:
def preview(path):
    if path.suffix.lower()==".csv": return pd.read_csv(path, nrows=5)
    if path.suffix.lower()==".xlsx": return pd.read_excel(path, nrows=5)
    return None
for path in sorted(DATA_DIR.iterdir()):
    frame=preview(path)
    if frame is None: continue
    for column in frame.select_dtypes(include="object"): frame[column]=frame[column].astype(str).str.replace(r"\\s+"," ",regex=True).str.slice(0,100)
    print(f"\n{path.name}: {frame.shape[1]} columns"); print(frame.to_string(index=False,max_cols=12))


cafe_transactions.xlsx: 10 columns
      Date Bill Number                       Item Desc     Time  Quantity  Rate   Tax  Discount  Total Category
2010-04-01     G0470115 QUA  MINERAL WATER(1000ML)     13:15:11         1    50 11.88         0  61.88 BEVERAGE
2010-04-01     G0470115 MONSOON MALABAR (AULAIT)       13:15:11         1   100 23.75         0 123.75 BEVERAGE
2010-04-01     G0470116 MASALA CHAI CUTTING            13:17:35         1    40  9.50         0  49.50 BEVERAGE
2010-04-01     G0470117 QUA  MINERAL WATER(1000ML)     13:19:55         1    50 11.88         0  61.88 BEVERAGE
2010-04-01     G0470283 MOROCCAN MINT TEA              01:20:18         1    45 10.69         0  55.69 BEVERAGE


## 5. Data-quality assessment

The pipeline checks missingness, duplicates, invalid fields, identifiers, cardinality, and problem-specific leakage or chronology risks. No row is silently removed.

## 6. Reusable implementation

Large functions are kept in source code so the notebook remains a readable analytical narrative.

In [4]:
source_path=PROJECT_ROOT/"src"/"analysis.py"; text=source_path.read_text(encoding="utf-8")
print(f"Reusable implementation: {len(text.splitlines())} lines")
print("Functions:", ", ".join(line.split("(")[0].replace("def ","").strip() for line in text.splitlines() if line.startswith("def ")))

Reusable implementation: 96 lines
Functions: run_analysis


## 7. Methodology and hypotheses

Duplicate control, monthly/category/daypart KPIs, order-value analysis, bill-level pair support/confidence/lift, and stable K-means menu-item segmentation.

The central hypothesis is that the audited features or group structure contain decision-relevant signal beyond the documented baseline. Exploratory findings are not presented as causal effects.

## 8. Execute the complete pipeline

This cell reruns cleaning, feature engineering, model/statistical analysis, validation, tables, figures, and model artifacts.

In [5]:
spec=importlib.util.spec_from_file_location("rebuilt_10_marketing_retail_analysis", PROJECT_ROOT/"src"/"analysis.py")
analysis=importlib.util.module_from_spec(spec); spec.loader.exec_module(analysis)
started=time.perf_counter(); results=analysis.run_analysis(); runtime=time.perf_counter()-started
assert results["status"]=="passed"
print(f"Pipeline status: {results['status']}\nRuntime: {runtime:.2f} seconds")

Pipeline status: passed
Runtime: 10.62 seconds


## 9. Executed data-quality evidence

In [6]:
for path in sorted((REPORTS_DIR/"tables").glob("*data_quality.csv")):
    frame=pd.read_csv(path); print(f"\n{path.name} ({len(frame)} fields)"); print(frame.to_string(index=False,max_rows=30))


data_quality.csv (10 fields)
      column          dtype  missing_count  missing_percent  unique_values  constant
        Date datetime64[ns]              0              0.0            365     False
Bill Number          object              0              0.0          69982     False
   Item Desc         object              0              0.0            580     False
        Time         object              0              0.0          36200     False
    Quantity          int64              0              0.0             20     False
        Rate        float64              0              0.0            134     False
         Tax        float64              0              0.0            445     False
    Discount        float64              0              0.0            111     False
       Total        float64              0              0.0            470     False
    Category         object              0              0.0              9     False


## 10. Baseline, candidates, and primary result

In [7]:
primary=REPORTS_DIR/"tables"/'basket_item_pairs.csv'
frame=pd.read_csv(primary); print(f"Primary evidence: {primary.name}, shape={frame.shape}")
print(frame.head(15).round(4).to_string(index=False))
print("\nVerified result:\n" + 'The cleaned year contains 69,982 bills and 32,654,640 recorded revenue units; two stable menu-item groups score 0.395 silhouette.')

Primary evidence: basket_item_pairs.csv, shape=(100, 7)
                        item_1                         item_2  joint_bills  support  confidence_1_to_2  confidence_2_to_1   lift
ADD FRIES                      B.M.T. PANINI                           151   0.0022             0.2008             0.0580 5.3943
B.M.T. PANINI                  MAGGI NDL ARRABIATA                     134   0.0019             0.0514             0.1521 4.0861
RED BULL 2+1                   SAMBUCA                                 290   0.0041             0.2485             0.0656 3.9318
MAGGI NDL ARRABIATA            SAMBUCA                                 175   0.0025             0.1986             0.0396 3.1429
CALCUTTA MINT                  RED BULL 2+1                            166   0.0024             0.0502             0.1422 3.0102
B.M.T. PANINI                  PHILLYCREAM CHEESE &CHILLY PAN          183   0.0026             0.0702             0.0984 2.6431
RED BULL ENERGY DRINK          SAMBUCA   

## 11. Validation, diagnostics, and robustness

In [8]:
sections=[key for key in ["validation","model_selection","tuning","residual_diagnostics","outlier_sensitivity","participant_bootstrap_intervals","diagnostic_90_percent_interval","empirical_90_percent_interval"] if key in results]
for key in sections: print(f"\n{key.upper()}\n"+json.dumps(results[key],indent=2)[:6000])

## 12. Visual evidence

### Cafe Business Evidence

![cafe_business_evidence](../reports/figures/cafe_business_evidence.png)

### Top Categories By Revenue

![top_categories_by_revenue](../reports/figures/top_categories_by_revenue.png)

## 13. Business interpretation

The cleaned year contains 69,982 bills and 32,654,640 recorded revenue units; two stable menu-item groups score 0.395 silhouette.

The correct action is to use this result as evidence for **Prioritize menu review, daypart staffing, and carefully tested cross-sell ideas.**, while retaining the documented baseline and monitoring the error or sensitivity segments.

## 14. Prioritized recommendations

1. Use the verified result to define a controlled follow-up rather than an automatic decision.
2. Monitor the weakest subgroup, time window, interval coverage, or cluster sensitivity shown in the saved tables.
3. Revalidate against a transparent baseline whenever the data or operating context changes.

## 15. Limitations, ethics, and responsible use

There is no customer identifier, so customer-level RFM, retention, and lifetime-value claims are not possible from this dataset.

Automated outputs remain associative unless a causal study design says otherwise.

## 16. Saved-artifact integrity

In [9]:
rows=[]
for path in sorted(REPORTS_DIR.rglob("*")):
    if path.is_file(): rows.append({"artifact": str(path.relative_to(PROJECT_ROOT)), "size_kb": round(path.stat().st_size/1000,1), "sha256": hashlib.sha256(path.read_bytes()).hexdigest()[:12]})
artifacts=pd.DataFrame(rows); print(artifacts.to_string(index=False,max_rows=80))

                                     artifact  size_kb       sha256
   reports/figures/cafe_business_evidence.png    265.4 83e377d5ed85
reports/figures/top_categories_by_revenue.png     46.6 7c25df62736e
                         reports/metrics.json      5.5 9a7f9ed54742
  reports/original_business_presentation.pptx   4168.6 a94b75a36f32
         reports/tables/basket_item_pairs.csv     14.8 e992128f4787
          reports/tables/category_summary.csv      0.3 edd8536db391
              reports/tables/data_quality.csv      0.4 08610720b449
           reports/tables/daypart_summary.csv      0.1 6821da985ffd
     reports/tables/menu_cluster_profiles.csv      0.1 ba91d70f2669
    reports/tables/menu_cluster_selection.csv      0.4 06328c9f160a
        reports/tables/menu_item_segments.csv     35.1 cb431097a700
  reports/tables/monthly_business_summary.csv      0.6 6874ad46629e


## 17. Acceptance check

In [10]:
metrics=json.loads((REPORTS_DIR/"metrics.json").read_text(encoding="utf-8"))
assert metrics["status"]=="passed"
assert list((REPORTS_DIR/"figures").glob("*.png"))
assert list((REPORTS_DIR/"tables").glob("*.csv"))
assert all(path.stat().st_size>0 for path in REPORTS_DIR.rglob("*") if path.is_file())
print("PASS: metrics status, figures, tables, and non-empty artifacts verified")

PASS: metrics status, figures, tables, and non-empty artifacts verified


## 18. Conclusion

The project addressed translate point-of-sale lines into revenue, order, daypart, basket, and stable menu-item insights supported by available fields. using duplicate control, monthly/category/daypart kpis, order-value analysis, bill-level pair support/confidence/lift, and stable k-means menu-item segmentation. The final verified conclusion is: **The cleaned year contains 69,982 bills and 32,654,640 recorded revenue units; two stable menu-item groups score 0.395 silhouette.** The next responsible step is external or current-data validation before operational use.

## 19. Reproduce locally

```bash
python projects/10-marketing-retail-analysis/src/analysis.py
python scripts/execute_notebooks.py --project 10-marketing-retail-analysis
```